In [40]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully")

Libraries imported successfully


In [41]:
def split_data (file_path,target_col="Owner",test_size=0.1,reandom_state=42):
    df=pd.read_csv(file_path)
    x=df.drop(target_col,axis=1)
    y=df[target_col]
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=test_size,random_state=reandom_state)
    return x_train,x_test,y_train,y_test
file_path=r"C:\Users\mosa\Downloads\cardata1.csv"
target_col="Owner"
x_train,x_test,y_train,y_test=split_data(file_path,target_col)
print(f"shape of train : {x_train.shape}")
print(f"shape of test : {y_train.shape}")

shape of train : (270, 8)
shape of test : (270,)


In [42]:
missing_percentage = (x_train.isna().sum() / len(x_train)) * 100
for col,val in missing_percentage.items():
    print(f"{col}: {val:.2f}%")

Car_Name: 0.37%
Year: 1.48%
Selling_Price: 2.96%
Present_Price: 1.85%
Kms_Driven: 1.85%
Fuel_Type: 0.74%
Seller_Type: 0.00%
Transmission: 0.37%


In [43]:
missing_percentage = (x_test.isna().sum() / len(x_train)) * 100
for col,val in missing_percentage.items():
    print(f"{col}: {val:.2f}%")

Car_Name: 0.00%
Year: 0.00%
Selling_Price: 0.37%
Present_Price: 0.37%
Kms_Driven: 0.37%
Fuel_Type: 0.00%
Seller_Type: 0.37%
Transmission: 0.00%


In [44]:
def cat_num_col(x_train):
    numerical_col=x_train.select_dtypes(include=["int32","float64"]).columns.tolist()
    categorical_col=x_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    return numerical_col,categorical_col

numerical_col,categorical_col=cat_num_col(x_train)
print(f" numerical_col: {numerical_col}")
print(f"categorical_col: {categorical_col}")

 numerical_col: ['Year', 'Selling_Price', 'Present_Price', 'Kms_Driven']
categorical_col: ['Car_Name', 'Fuel_Type', 'Seller_Type', 'Transmission']


In [45]:
def get_num_cat(x_train,x_test,numerical_col,categorical_col):
    x_train_num=x_train[numerical_col].copy()
    x_test_num=x_test[numerical_col].copy()
    x_train_cat=x_train[categorical_col].copy()
    x_test_cat=x_test[categorical_col].copy()
    return x_train_num,x_test_num,x_train_cat,x_test_cat

x_train_num,x_test_num,x_train_cat,x_test_cat=get_num_cat(x_train,x_test,numerical_col,categorical_col)
print(f"size of x_train numerical : {x_train_num.shape}")
print(f"size of x_test numerical  : {x_test_num.shape}")
print(f"size of x_train categorical : {x_train_cat.shape}")
print(f"size of x_test categorical   : {x_test_cat.shape}")

size of x_train numerical : (270, 4)
size of x_test numerical  : (31, 4)
size of x_train categorical : (270, 4)
size of x_test categorical   : (31, 4)


In [46]:
def fill_missing_num(x_train_num,x_test_num,strayigy="median"):
    col_name=x_train_num.columns
    x_train_imp=x_train_num.copy().values
    x_test_imp=x_test_num.copy().values

    print(f"missing x_train data before using stratigy : {np.sum(pd.isnull(x_train_imp))}")
    print(f"missing x_test data before using stratigy : {np.sum(pd.isnull(x_test_imp))}")

    if strayigy=="median":
        imput_val=np.nanmedian(x_train_num,axis=0)
        print("you use the tecnic median ")

    elif strayigy=="mean":
        imput_val=np.nanmean(x_train_num,axis=0)
        print("you use the tecnic mean ")

    for col in range(x_train_num.shape[1]):
        missing_mask_train=pd.isnull(x_train_imp[:,col])
        x_train_imp[missing_mask_train,col]=imput_val[col]

        missing_mask_test=pd.isnull(x_test_imp[:,col])
        x_test_imp[missing_mask_test,col]=imput_val[col]

    print(f"missing x_train data after using stratigy : {np.sum(pd.isnull(x_train_imp))}")
    print(f"missing x_test data after using stratigy: {np.sum(pd.isnull(x_test_imp))}")  

    x_train_imp=pd.DataFrame(x_train_imp,columns=col_name)
    x_test_imp=pd.DataFrame(x_test_imp,columns=col_name)

    return x_train_imp,x_test_imp

x_train_num,x_test_num=fill_missing_num(x_train_num,x_test_num,strayigy="median")



missing x_train data before using stratigy : 22
missing x_test data before using stratigy : 3
you use the tecnic median 
missing x_train data after using stratigy : 0
missing x_test data after using stratigy: 0
